In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

c:\Users\hp\Documents\Anemia_Fusion_Net_project


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, random_split

from ml.datasets.clinical_dataset import ClinicalDataset
from ml.models.clinical_model import ClinicalModel

from ml.config import *

PROJECT_ROOT : C:\Users\hp\Documents\Anemia_Fusion_Net_project
TRAIN_CSV    : C:\Users\hp\Documents\Anemia_Fusion_Net_project\data\processed\train.csv
VAL_CSV      : C:\Users\hp\Documents\Anemia_Fusion_Net_project\data\processed\valid.csv
IMAGE_DIR    : C:\Users\hp\Documents\Anemia_Fusion_Net_project\data\eye_image
Device       : cpu


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("Clinical Model Training")
print("=" * 70)

print("Device :", device)

print("\nConfiguration")
print("-" * 70)

print("TRAIN_CSV :", TRAIN_CSV)
print("VAL_CSV   :", VAL_CSV)
print("TEST_CSV  :", TEST_CSV)

Clinical Model Training
Device : cpu

Configuration
----------------------------------------------------------------------
TRAIN_CSV : C:\Users\hp\Documents\Anemia_Fusion_Net_project\data\processed\train.csv
VAL_CSV   : C:\Users\hp\Documents\Anemia_Fusion_Net_project\data\processed\valid.csv
TEST_CSV  : C:\Users\hp\Documents\Anemia_Fusion_Net_project\data\processed\test.csv


In [4]:
from ml.config import *

print(dir())

['BATCH_SIZE', 'ClinicalDataset', 'ClinicalModel', 'DATA_DIR', 'DEVICE', 'DataLoader', 'IMAGE_DIR', 'IMAGE_SIZE', 'In', 'LEARNING_RATE', 'MODEL_DIR', 'NUM_CLASSES', 'NUM_EPOCHS', 'NUM_WORKERS', 'OUTPUT_DIR', 'Out', 'PATIENCE', 'PROCESSED_DIR', 'PROJECT_ROOT', 'Path', 'TEST_CSV', 'TRAIN_CSV', 'VAL_CSV', 'WEIGHT_DECAY', '_', '__', '___', '__builtin__', '__builtins__', '__doc__', '__loader__', '__name__', '__package__', '__spec__', '__vsc_ipynb_file__', '_dh', '_i', '_i1', '_i2', '_i3', '_i4', '_ih', '_ii', '_iii', '_oh', 'device', 'exit', 'get_ipython', 'nn', 'open', 'optim', 'quit', 'random_split', 'sys', 'torch']


In [6]:
import inspect
from ml.datasets.clinical_dataset import ClinicalDataset

print(inspect.signature(ClinicalDataset.__init__))

(self, csv_file, train=True)


In [9]:
from pathlib import Path

CLINICAL_CSV = PROJECT_ROOT / "data" / "clinical" / "anemia.csv"

print(CLINAL_CSV if False else CLINICAL_CSV)

dataset = ClinicalDataset(
    csv_file=CLINICAL_CSV,
    train=True
)

print("=" * 70)
print("Clinical Dataset Loaded Successfully")
print("=" * 70)

print("Total Samples :", len(dataset))

c:\Users\hp\Documents\Anemia_Fusion_Net_project\data\clinical\anemia.csv
Clinical Dataset Loaded Successfully
Total Samples : 1136


In [8]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

for f in PROJECT_ROOT.rglob("*.csv"):
    print(f)

c:\Users\hp\Documents\Anemia_Fusion_Net_project\data\clinical\anemia.csv
c:\Users\hp\Documents\Anemia_Fusion_Net_project\data\geo\datafile.csv
c:\Users\hp\Documents\Anemia_Fusion_Net_project\data\geo\geo_risk.csv
c:\Users\hp\Documents\Anemia_Fusion_Net_project\data\processed\fusion_data.csv
c:\Users\hp\Documents\Anemia_Fusion_Net_project\data\processed\image_metadata.csv
c:\Users\hp\Documents\Anemia_Fusion_Net_project\data\processed\test.csv
c:\Users\hp\Documents\Anemia_Fusion_Net_project\data\processed\train.csv
c:\Users\hp\Documents\Anemia_Fusion_Net_project\data\processed\valid.csv


In [10]:
clinical, label = dataset[0]

print("=" * 70)
print("Sample Loaded")
print("=" * 70)

print("Clinical Features :", clinical)
print("Shape :", clinical.shape)
print("Label :", label)

Sample Loaded
Clinical Features : tensor([ 0.9593, -1.2224,  1.6619,  0.3205, -1.3416])
Shape : torch.Size([5])
Label : tensor(1)


In [11]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print("=" * 70)
print("Dataset Split Successfully")
print("=" * 70)

print("Training Samples  :", len(train_dataset))
print("Validation Samples:", len(val_dataset))

Dataset Split Successfully
Training Samples  : 908
Validation Samples: 228


In [12]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

print("=" * 70)
print("DataLoaders Ready")
print("=" * 70)

print("Train Batches :", len(train_loader))
print("Validation Batches :", len(val_loader))

DataLoaders Ready
Train Batches : 29
Validation Batches : 8


In [13]:
clinical, labels = next(iter(train_loader))

print("=" * 70)
print("One Batch Loaded")
print("=" * 70)

print("Clinical Shape :", clinical.shape)
print("Labels Shape   :", labels.shape)

One Batch Loaded
Clinical Shape : torch.Size([32, 5])
Labels Shape   : torch.Size([32])


In [14]:
model = ClinicalModel().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

print("=" * 70)
print("Clinical Model Loaded Successfully")
print("=" * 70)

print("Loss Function :", criterion)
print("Optimizer     :", optimizer.__class__.__name__)

Clinical Model Loaded Successfully
Loss Function : CrossEntropyLoss()
Optimizer     : AdamW


In [15]:
clinical = clinical.to(device)
labels = labels.to(device)

model.eval()

with torch.no_grad():
    outputs = model(clinical)

print("=" * 70)
print("Forward Pass Successful")
print("=" * 70)

print("Input Shape  :", clinical.shape)
print("Output Shape :", outputs.shape)

Forward Pass Successful
Input Shape  : torch.Size([32, 5])
Output Shape : torch.Size([32, 2])


In [16]:
loss = criterion(outputs, labels)

print("=" * 70)
print("Loss")
print("=" * 70)

print(loss.item())

Loss
0.7491151690483093


In [17]:
predictions = torch.argmax(outputs, dim=1)

print("=" * 70)
print("Predictions")
print("=" * 70)

print(predictions)

Predictions
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1])


In [18]:
print("=" * 70)
print("Notebook 11 Completed Successfully")
print("=" * 70)

print("✓ Clinical Dataset Loaded")
print("✓ DataLoader Created")
print("✓ Clinical Model Loaded")
print("✓ Forward Pass Verified")
print("✓ Loss Computed")
print("✓ Predictions Generated")

print("\nNext Notebook: 12_train_geo.ipynb")

Notebook 11 Completed Successfully
✓ Clinical Dataset Loaded
✓ DataLoader Created
✓ Clinical Model Loaded
✓ Forward Pass Verified
✓ Loss Computed
✓ Predictions Generated

Next Notebook: 12_train_geo.ipynb
